# Phase 3 — Reading & Writing Data

**Duration:** Week 5 | **Goal:** Master every file format and I/O pattern you'll use daily.

---

### What We'll Cover

| Area | Key Skills |
| --- | --- |
| File Formats | CSV, JSON, Parquet, Delta, ORC — when to use each |
| Reading Data | `spark.read` options, schemas, corrupt record handling |
| Writing Data | `spark.write`, save modes (overwrite/append/ignore/error), partitioning on disk |
| Delta Lake Intro | ACID transactions, time travel, versioning |
| Schema Evolution | `mergeSchema`, schema enforcement, handling new columns |
| Auto Loader | Incremental file ingestion with `cloudFiles` |

### Dataset

* **Primary:** `samples.tpch.lineitem` (30M rows) — we'll use subsets for format experiments
* **Volume:** `arnalladatabricks.pyspark_learning.raw_data` — read/write destination

---

**Pre-requisite:** Phase 2 completed. You should be comfortable with `select`, `filter`, `withColumn`, `groupBy`, and joins.

In [0]:
# ============================================================
# STEP 1: Setup — load base dataset and confirm volume access
# ============================================================
from pyspark.sql.functions import col, count

# Our working volume (created in Phase 2)
VOLUME_PATH = "/Volumes/arnalladatabricks/pyspark_learning/raw_data"

# Load our dataset
df = spark.read.table("samples.tpch.lineitem")

# Create a small subset (20K rows) for quick format experiments
df_small = df.limit(20_000)

print(f"Full dataset:  {df.count():,} rows, {len(df.columns)} columns")
print(f"Small subset:  20,000 rows (for format comparisons)")
print(f"\nVolume path: {VOLUME_PATH}")
print("\n✅ Ready for Phase 3!")

## 3.1 — File Formats: The Big Picture

All data lives in files. The format you choose affects **speed**, **storage cost**, and **capabilities**.

| Format | Type | Compression | Schema | Splittable | Best For |
| --- | --- | --- | --- | --- | --- |
| **CSV** | Row-based, text | None (or gzip) | No (inferred) | Yes (unless gzipped) | Legacy interchange, Excel exports |
| **JSON** | Row-based, text | None (or gzip) | Partial (self-describing) | Yes (line-delimited) | APIs, logs, semi-structured |
| **Parquet** | Columnar, binary | Snappy/Zstd (built-in) | Yes (embedded) | Yes | Analytics, data lakes (DEFAULT) |
| **Delta** | Parquet + transaction log | Same as Parquet | Yes + enforced | Yes | Production tables, ACID, time travel |
| **ORC** | Columnar, binary | Zlib/Snappy | Yes (embedded) | Yes | Hive ecosystem (legacy) |

### Key Insight: Row-based vs Columnar

* **Row-based (CSV, JSON):** Each row stored together. Good for writing all columns. Bad for reading a few columns from wide tables.
* **Columnar (Parquet, Delta, ORC):** Each COLUMN stored together. Reading 3 columns from a 100-column table? Only those 3 columns are scanned. **Massively faster for analytics.**

### The Rule of Thumb

* **Receiving data?** Accept whatever format the source sends (CSV from vendors, JSON from APIs)
* **Storing data?** Always convert to **Delta** (or at minimum Parquet)
* **Never store production data as CSV/JSON** — slow reads, no schema, no ACID

### Deep Dive: Why Format Matters So Much

**Real-world scenario:** You have 100 GB of sales data with 50 columns. Your dashboard only needs 3 columns (date, region, revenue).

| Format | Data scanned | Why |
| --- | --- | --- |
| CSV | 100 GB (all of it) | Row-based: must read every column to find your 3 |
| Parquet | ~6 GB (just 3 columns) | Columnar: skips the other 47 columns entirely |
| Delta | ~6 GB + predicate pushdown | Same as Parquet, plus skips entire files via stats |

That’s a **16x** reduction in I/O just by choosing the right format.

**Why Delta over plain Parquet?**

Parquet files are just files — no transaction log. Problems:
* Two jobs writing at the same time? Corrupt data.
* Reader sees a half-written file? Partial results.
* Need to undo yesterday’s bad write? Impossible.

Delta adds a **transaction log** (`_delta_log/`) that tracks every change:
* **ACID** — writes are atomic (all-or-nothing)
* **Time travel** — query any previous version
* **Schema enforcement** — rejects writes that don’t match the schema
* **MERGE/UPDATE/DELETE** — row-level operations (Parquet is append-only)

**CSV/JSON: When you’d still use them:**
* Receiving data from external systems (they don’t speak Parquet)
* One-time imports (load CSV, convert to Delta, delete the CSV)
* Human-readable debugging (open in Excel or text editor)

In [0]:
# ============================================================
# STEP 2: Write the SAME data in 4 formats and compare
# ============================================================
import time

# Use our small subset (20K rows) for quick format comparisons
base_path = f"{VOLUME_PATH}/format_comparison"

# --- Write as CSV ---
start = time.time()
df_small.write.mode("overwrite").option("header", "true").csv(f"{base_path}/csv")
csv_time = time.time() - start

# --- Write as JSON ---
start = time.time()
df_small.write.mode("overwrite").json(f"{base_path}/json")
json_time = time.time() - start

# --- Write as Parquet ---
start = time.time()
df_small.write.mode("overwrite").parquet(f"{base_path}/parquet")
parquet_time = time.time() - start

# --- Write as Delta ---
start = time.time()
df_small.write.mode("overwrite").format("delta").save(f"{base_path}/delta")
delta_time = time.time() - start

print("WRITE TIMES (20K rows):")
print(f"  CSV:     {csv_time:.2f}s")
print(f"  JSON:    {json_time:.2f}s")
print(f"  Parquet: {parquet_time:.2f}s")
print(f"  Delta:   {delta_time:.2f}s")
print("\n→ Write speeds are similar for small data.")
print("  The differences show up on READ and at scale.")

In [0]:
# ============================================================
# STEP 3: Read back from each format — compare speed + features
# ============================================================
import time

base_path = f"{VOLUME_PATH}/format_comparison"

# --- Read CSV (needs header option, infers schema) ---
start = time.time()
df_csv = spark.read.option("header", "true").option("inferSchema", "true").csv(f"{base_path}/csv")
df_csv.count()
csv_read = time.time() - start

# --- Read JSON ---
start = time.time()
df_json = spark.read.json(f"{base_path}/json")
df_json.count()
json_read = time.time() - start

# --- Read Parquet (schema embedded, no options needed) ---
start = time.time()
df_parquet = spark.read.parquet(f"{base_path}/parquet")
df_parquet.count()
parquet_read = time.time() - start

# --- Read Delta ---
start = time.time()
df_delta = spark.read.format("delta").load(f"{base_path}/delta")
df_delta.count()
delta_read = time.time() - start

print("READ TIMES (20K rows):")
print(f"  CSV:     {csv_read:.2f}s  (schema inferred = extra scan)")
print(f"  JSON:    {json_read:.2f}s  (schema inferred)")
print(f"  Parquet: {parquet_read:.2f}s  (schema embedded!)")
print(f"  Delta:   {delta_read:.2f}s  (schema embedded + transaction log)")

print("\n--- SCHEMA COMPARISON ---")
print("CSV schema (inferred):")
df_csv.printSchema()
print("Parquet schema (exact, embedded):")
df_parquet.printSchema()

### Deep Dive: What the Results Tell Us

**Read speed ranking:** Delta (0.39s) > Parquet (0.98s) > JSON (1.06s) > CSV (2.70s)

Why Delta beat Parquet here:
* Delta’s transaction log contains **file-level statistics** (min/max values per column per file). Even for a simple `count()`, Delta knows the row count from metadata — no full scan needed.
* Parquet must open each file to count rows.

**The CSV schema problem (look at Step 3 output):**

Original data has `l_orderkey` as `LongType` and `l_quantity` as `DecimalType(18,2)`. But CSV inferred:
* `l_orderkey` as `IntegerType` — will OVERFLOW for values > 2 billion
* `l_quantity` as `DoubleType` — loses exact decimal precision (money!)

This is exactly why **explicit schemas beat inferSchema** in production. CSV loses type information on write; you get it back wrong on read.

**Key syntax patterns from Steps 2-3:**

| Code | What it does |
| --- | --- |
| `.write.mode("overwrite")` | Replace existing files at that path |
| `.option("header", "true")` | Include column names as first row (CSV) |
| `.format("delta").save(path)` | Write as Delta (alternative: `.write.delta(path)`) |
| `.read.option("inferSchema", "true")` | Guess types by scanning data (slow, error-prone) |
| `.read.parquet(path)` | Read Parquet (schema embedded, no options needed!) |
| `.read.format("delta").load(path)` | Read Delta from a file path |

## 3.2 — Reading Data: CSV & JSON Options

### CSV Read Options (the messy reality)

CSV files from vendors are rarely clean. Common issues:
* No header row, or wrong header
* Different delimiters (`;`, `|`, `\t`)
* Quoted fields with commas inside
* Corrupt/malformed rows
* Encoding issues (UTF-8 vs Latin-1)

```python
spark.read \
    .option("header", "true")          # first row = column names
    .option("inferSchema", "true")      # guess types (slow; use explicit schema)
    .option("sep", ",")                 # delimiter (default comma)
    .option("quote", '"')               # quote character
    .option("escape", "\\")             # escape character
    .option("multiLine", "true")        # allow fields spanning lines
    .option("encoding", "UTF-8")        # character encoding
    .option("nullValue", "NA")          # treat "NA" as null
    .option("mode", "PERMISSIVE")       # how to handle corrupt rows
    .csv("path/to/files/")
```

### Read Modes (how to handle bad rows)

| Mode | Behavior | Use When |
| --- | --- | --- |
| `PERMISSIVE` (default) | Puts corrupt rows in `_corrupt_record` column, nulls for bad fields | Production — don’t crash, inspect later |
| `DROPMALFORMED` | Silently drops bad rows | Quick exploration (caution: data loss) |
| `FAILFAST` | Throws exception on first bad row | Testing — catch problems immediately |

In [0]:
# ============================================================
# STEP 4: Reading CSV with various options
# ============================================================
from pyspark.sql.functions import col, count, when

base_path = f"{VOLUME_PATH}/format_comparison"

# --- Default read (header + inferSchema) ---
print("1. Default CSV read (infers types):")
df_default = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(f"{base_path}/csv")
df_default.select("l_orderkey", "l_quantity", "l_extendedprice").show(3)
print(f"   Schema: l_orderkey={df_default.schema['l_orderkey'].dataType}")
print(f"   Schema: l_quantity={df_default.schema['l_quantity'].dataType}")

# --- Read with EXPLICIT schema (correct types, fast) ---
print("\n2. CSV read with explicit schema (correct types, no extra scan):")
from pyspark.sql.types import (
    StructType, StructField, LongType, IntegerType,
    DecimalType, StringType, DateType
)

# Provide the correct types ourselves
csv_schema = StructType([
    StructField("l_orderkey", LongType()),
    StructField("l_partkey", LongType()),
    StructField("l_suppkey", LongType()),
    StructField("l_linenumber", IntegerType()),
    StructField("l_quantity", DecimalType(18, 2)),
    StructField("l_extendedprice", DecimalType(18, 2)),
    StructField("l_discount", DecimalType(18, 2)),
    StructField("l_tax", DecimalType(18, 2)),
    StructField("l_returnflag", StringType()),
    StructField("l_linestatus", StringType()),
    StructField("l_shipdate", DateType()),
    StructField("l_commitdate", DateType()),
    StructField("l_receiptdate", DateType()),
    StructField("l_shipinstruct", StringType()),
    StructField("l_shipmode", StringType()),
    StructField("l_comment", StringType()),
])

import time
start = time.time()
df_explicit = spark.read \
    .option("header", "true") \
    .schema(csv_schema) \
    .csv(f"{base_path}/csv")
df_explicit.count()
explicit_time = time.time() - start

print(f"   Schema: l_orderkey={df_explicit.schema['l_orderkey'].dataType} ✅ (correct!)")
print(f"   Schema: l_quantity={df_explicit.schema['l_quantity'].dataType} ✅ (correct!)")
print(f"   Read time with explicit schema: {explicit_time:.2f}s (no inferSchema scan)")

In [0]:
# ============================================================
# STEP 5: Handling corrupt/malformed CSV data
# ============================================================
from pyspark.sql.functions import col, count
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

# Create a deliberately messy CSV-like dataset
messy_data = [
    ("1,Alice,50000.00",),    # good
    ("2,Bob,60000.00",),      # good
    ("bad_data_here",),        # corrupt: wrong number of fields
    ("4,Diana,abc",),          # corrupt: 'abc' is not a number
    ("5,Eve,75000.50",),       # good
]
df_messy_raw = spark.createDataFrame(messy_data, ["raw_line"])

# Write as a raw text file to simulate a messy CSV
messy_path = f"{VOLUME_PATH}/messy_data.csv"
# Write a proper CSV with known bad rows
from pyspark.sql.functions import lit
lines = ["id,name,salary", "1,Alice,50000.00", "2,Bob,60000.00", "bad_data_here", "4,Diana,abc", "5,Eve,75000.50"]
df_lines = spark.createDataFrame([(l,) for l in lines[1:]], ["value"])  # skip header

# Use createDataFrame with the bad data directly
import pandas as pd
pdf = pd.DataFrame({
    "id": ["1", "2", "bad", "4", "5"],
    "name": ["Alice", "Bob", "data_here", "Diana", "Eve"],
    "salary": ["50000.00", "60000.00", "", "abc", "75000.50"]
})
df_messy_spark = spark.createDataFrame(pdf)
df_messy_spark.write.mode("overwrite").option("header", "true").csv(f"{VOLUME_PATH}/messy_employees")

# --- PERMISSIVE mode (default): nulls for bad fields ---
print("PERMISSIVE mode (default) — corrupt fields become null:")
schema = StructType([
    StructField("id", IntegerType()),
    StructField("name", StringType()),
    StructField("salary", DoubleType()),
])
df_perm = spark.read \
    .option("header", "true") \
    .option("mode", "PERMISSIVE") \
    .schema(schema) \
    .csv(f"{VOLUME_PATH}/messy_employees")
df_perm.show()

# Count the nulls (these are our corrupt records)
null_count = df_perm.filter(col("id").isNull() | col("salary").isNull()).count()
print(f"→ {null_count} rows have null values (corrupt or unparseable fields)")

# --- DROPMALFORMED mode: bad rows vanish ---
print("\nDROPMALFORMED mode — bad rows silently removed:")
df_drop = spark.read \
    .option("header", "true") \
    .option("mode", "DROPMALFORMED") \
    .schema(schema) \
    .csv(f"{VOLUME_PATH}/messy_employees")
df_drop.show()
print(f"→ {df_drop.count()} rows kept (bad rows dropped silently)")

## 3.3 — Writing Data: Save Modes & Partitioning

### Save Modes

When writing data, what happens if files already exist at the destination?

| Mode | Behavior | Use When |
| --- | --- | --- |
| `"error"` (default) | Throws exception if path exists | Safety net — prevent accidental overwrites |
| `"overwrite"` | Deletes existing data, writes new | Rebuilding a table from scratch |
| `"append"` | Adds new files alongside existing | Incremental loads, streaming sinks |
| `"ignore"` | Does nothing if path exists | Idempotent jobs (safe to re-run) |

### Partitioning on Disk

Write data split into folders by a column’s values:
```python
df.write.partitionBy("year", "month").parquet("path/")
```
Creates: `path/year=2024/month=01/data.parquet`, `path/year=2024/month=02/data.parquet`, etc.

**Why partition?** When you filter `WHERE year = 2024`, Spark only reads the `year=2024/` folder — skips everything else (**partition pruning**).

**When to partition:**
* Column has LOW cardinality (year, month, country — NOT customer_id)
* Queries ALWAYS filter on that column
* Each partition has enough data (>100 MB ideally)

### coalesce vs repartition (controlling file count)

| Method | What it does | Shuffle? |
| --- | --- | --- |
| `repartition(n)` | Redistributes to exactly N partitions | Yes (expensive) |
| `coalesce(n)` | Reduces to N partitions (combine only) | No (cheap, narrow) |

Use `coalesce(1)` when you want a single output file (small data). Use `repartition` when you need even distribution.

In [0]:
# ============================================================
# STEP 6: Save modes and disk partitioning
# ============================================================
from pyspark.sql.functions import col, year, month

base_path = f"{VOLUME_PATH}/write_demos"

# --- SAVE MODES ---
print("SAVE MODES DEMO:")
print("="*50)

# 1. Overwrite: replace everything
df_small.write.mode("overwrite").parquet(f"{base_path}/overwrite_test")
print("1. mode('overwrite') — wrote 20K rows")

# 2. Append: add more data alongside existing
df_small.write.mode("append").parquet(f"{base_path}/overwrite_test")
count_after_append = spark.read.parquet(f"{base_path}/overwrite_test").count()
print(f"2. mode('append') — now {count_after_append:,} rows (doubled!)")

# 3. Ignore: do nothing if exists
df_small.write.mode("ignore").parquet(f"{base_path}/overwrite_test")
count_after_ignore = spark.read.parquet(f"{base_path}/overwrite_test").count()
print(f"3. mode('ignore') — still {count_after_ignore:,} rows (did nothing)")

# 4. Overwrite again to reset
df_small.write.mode("overwrite").parquet(f"{base_path}/overwrite_test")
count_after_reset = spark.read.parquet(f"{base_path}/overwrite_test").count()
print(f"4. mode('overwrite') again — back to {count_after_reset:,} rows")

# --- PARTITION BY ---
print("\n\nPARTITION BY DEMO:")
print("="*50)

# Partition by ship mode (7 distinct values)
df_small.write.mode("overwrite") \
    .partitionBy("l_shipmode") \
    .parquet(f"{base_path}/partitioned_by_shipmode")

# Read back — show that partition pruning works
df_partitioned = spark.read.parquet(f"{base_path}/partitioned_by_shipmode")
print(f"Total rows: {df_partitioned.count():,}")
print(f"AIR only:   {df_partitioned.filter(col('l_shipmode') == 'AIR').count():,}")
print("\n→ When you filter by l_shipmode, Spark reads ONLY that folder!")

# --- COALESCE: Control output file count ---
print("\n\nCOALESCE DEMO:")
print("="*50)

# Single file output (useful for small data)
df_small.coalesce(1).write.mode("overwrite").parquet(f"{base_path}/single_file")
print("coalesce(1) — wrote as a single Parquet file")
print("→ Good for: small exports, sharing with non-Spark tools")
print("→ Bad for: large data (one file = one task = no parallelism)")

### Deep Dive: Writing Patterns in Production

**The append trap:**

If your job fails halfway and you re-run it with `mode("append")`, you get DUPLICATE data. This is why Delta Lake exists — it gives you:
* `mode("overwrite")` that’s atomic (old data visible until new data is fully written)
* MERGE (upsert) to handle duplicates intelligently

**Partition column choices — real examples:**

| Good partitioning | Bad partitioning |
| --- | --- |
| `partitionBy("year", "month")` — 12-60 folders | `partitionBy("customer_id")` — millions of folders! |
| `partitionBy("country")` — ~200 folders | `partitionBy("order_id")` — one file per order (tiny!) |
| `partitionBy("event_date")` — 365 folders/year | `partitionBy("timestamp")` — infinite folders |

**The “small files problem”:** Too many partitions = too many tiny files = slow metadata operations. Rule: each partition should have at least 100 MB of data.

**coalesce(1) — when and when NOT:**
* ✅ Small lookup tables, config files, exports to non-Spark consumers
* ❌ Large datasets — one file means one task, no parallelism on read
* Compromise: `coalesce(8)` for medium data — fewer files but still parallel

**What does `mode("overwrite")` actually do with Parquet?**

It DELETES the entire directory, then writes new files. If your job crashes mid-write, you lose both old AND new data. Delta’s overwrite is safe — it uses the transaction log to atomically swap versions.

## 3.4 — Delta Lake: Your Production Format

Delta Lake = Parquet files + a transaction log. It’s the DEFAULT format for all tables in Databricks.

### What Delta Gives You (that Parquet doesn’t)

| Feature | Plain Parquet | Delta Lake |
| --- | --- | --- |
| ACID transactions | ❌ | ✅ Atomic writes, no partial reads |
| Time travel | ❌ | ✅ Query any previous version |
| UPDATE / DELETE / MERGE | ❌ (append-only) | ✅ Row-level operations |
| Schema enforcement | ❌ | ✅ Rejects bad schemas on write |
| Schema evolution | Manual | ✅ `mergeSchema` option |
| Audit history | ❌ | ✅ Full change log |
| OPTIMIZE / Z-ORDER | ❌ | ✅ File compaction + co-location |

### Key Commands
```python
# Write as Delta
df.write.format("delta").save("/path")              # to file path
df.write.saveAsTable("catalog.schema.table_name")    # to Unity Catalog (preferred)

# Read Delta
df = spark.read.format("delta").load("/path")
df = spark.read.table("catalog.schema.table_name")

# Time travel
df_v0 = spark.read.format("delta").option("versionAsOf", 0).load("/path")
df_yesterday = spark.read.format("delta").option("timestampAsOf", "2024-01-01").load("/path")

# History
spark.sql("DESCRIBE HISTORY catalog.schema.table_name")
```

In [0]:
# ============================================================
# STEP 7: Delta Lake — ACID, time travel, history
# ============================================================
from pyspark.sql.functions import col, lit
from delta.tables import DeltaTable

delta_path = f"{VOLUME_PATH}/delta_demo"

# --- Write initial version (v0) ---
df_small.write.mode("overwrite").format("delta").save(delta_path)
print("Version 0: Wrote 20,000 rows")
print(f"  Row count: {spark.read.format('delta').load(delta_path).count():,}")

# --- Append more data (v1) ---
df_extra = df.limit(5_000)  # 5K more rows
df_extra.write.mode("append").format("delta").save(delta_path)
print(f"\nVersion 1: Appended 5,000 rows")
print(f"  Row count: {spark.read.format('delta').load(delta_path).count():,}")

# --- Overwrite (v2) ---
df_small.write.mode("overwrite").format("delta").save(delta_path)
print(f"\nVersion 2: Overwrote with 20,000 rows")
print(f"  Row count: {spark.read.format('delta').load(delta_path).count():,}")

# --- TIME TRAVEL: read previous versions ---
print("\n" + "="*50)
print("TIME TRAVEL:")
print("="*50)

df_v0 = spark.read.format("delta").option("versionAsOf", 0).load(delta_path)
df_v1 = spark.read.format("delta").option("versionAsOf", 1).load(delta_path)
df_v2 = spark.read.format("delta").option("versionAsOf", 2).load(delta_path)

print(f"  Version 0: {df_v0.count():,} rows (initial write)")
print(f"  Version 1: {df_v1.count():,} rows (after append)")
print(f"  Version 2: {df_v2.count():,} rows (after overwrite)")
print("\n→ Every version is still accessible! Nothing is lost.")

# --- HISTORY: see the audit log ---
print("\nDELTA HISTORY (what happened and when):")
dt = DeltaTable.forPath(spark, delta_path)
display(dt.history().select("version", "timestamp", "operation", "operationMetrics"))

### Deep Dive: How Delta Lake Actually Works

**Under the hood:** A Delta table is just a folder with:
```
/delta_demo/
  _delta_log/          ← Transaction log (JSON files)
    00000000000000000000.json   ← Version 0: "added file1.parquet, file2.parquet"
    00000000000000000001.json   ← Version 1: "added file3.parquet"
    00000000000000000002.json   ← Version 2: "removed file1-3, added file4.parquet"
  part-00000-xxx.parquet   ← Actual data files
  part-00001-xxx.parquet
```

**Time travel works because:** Old Parquet files aren’t deleted immediately. The log says which files belong to each version. Reading v0 = read files listed in log entry 0.

**VACUUM** cleans up: `VACUUM delta_path RETAIN 168 HOURS` deletes files no longer referenced by any version within 7 days. After vacuum, old versions become inaccessible.

**Real-world time travel uses:**
* “What did the data look like before yesterday’s ETL ran?”
* “Someone corrupted the table — restore to last good version”
* “Audit: prove what data was used for the Q3 report”
* ML reproducibility: “retrain on the exact same data snapshot”

**saveAsTable vs save(path):**

| Method | What it does |
| --- | --- |
| `df.write.saveAsTable("catalog.schema.name")` | Registers in Unity Catalog (discoverable, governed, preferred) |
| `df.write.format("delta").save("/path")` | Just writes files (unregistered, not governed) |

In production, always use `saveAsTable` — it gives you UC governance, permissions, lineage tracking.

## 3.5 — Schema Evolution

### The Problem

Data sources change over time:
* Vendor adds a new column to their CSV export
* API v2 adds fields that v1 didn’t have
* Upstream team renames a column

What happens when you try to write data with a DIFFERENT schema to an existing Delta table?

### Schema Enforcement (default behavior)

By default, Delta **rejects** writes that don’t match the existing schema:
* New column in data? ❌ Error
* Missing column? ❌ Error
* Type mismatch? ❌ Error

This is a SAFETY feature — prevents accidentally corrupting production tables.

### Schema Evolution (opt-in)

When you WANT to accept new columns:
```python
df_new.write \
    .mode("append") \
    .option("mergeSchema", "true") \
    .format("delta") \
    .save(delta_path)
```

`mergeSchema = true` tells Delta: “it’s okay, add the new columns to the table schema.”

### What mergeSchema handles:

| Change | Behavior |
| --- | --- |
| New column added | ✅ Column added; old rows get NULL for it |
| Column removed from data | ✅ Old column stays; new rows get NULL |
| Type widening (int → long) | ✅ Automatically widened |
| Type narrowing (long → int) | ❌ Rejected (lossy) |
| Column renamed | ❌ Treated as drop + add (not detected as rename) |

In [0]:
# ============================================================
# STEP 8: Schema evolution in action
# ============================================================
from pyspark.sql.functions import col, lit
from pyspark.sql.types import StructType, StructField, LongType, StringType, DecimalType

evolution_path = f"{VOLUME_PATH}/schema_evolution_demo"

# --- Create initial table (3 columns) ---
df_v1 = spark.createDataFrame([
    (1, "Alice", 50000.00),
    (2, "Bob", 60000.00),
    (3, "Charlie", 55000.00),
], ["id", "name", "salary"])

df_v1.write.mode("overwrite").format("delta").save(evolution_path)
print("Initial table (3 columns):")
spark.read.format("delta").load(evolution_path).show()
spark.read.format("delta").load(evolution_path).printSchema()

# --- Try to append with a NEW column (without mergeSchema) ---
df_v2 = spark.createDataFrame([
    (4, "Diana", 70000.00, "Engineering"),
    (5, "Eve", 65000.00, "Marketing"),
], ["id", "name", "salary", "department"])

print("\nTrying to append with a new 'department' column...")
try:
    df_v2.write.mode("append").format("delta").save(evolution_path)
    print("✅ Write succeeded (shouldn't happen without mergeSchema)")
except Exception as e:
    error_msg = str(e)[:200]
    print(f"❌ REJECTED! Schema enforcement blocked the write.")
    print(f"   Error: {error_msg}...")

# --- Now with mergeSchema = true ---
print("\nRetrying with mergeSchema=true...")
df_v2.write.mode("append") \
    .option("mergeSchema", "true") \
    .format("delta") \
    .save(evolution_path)

print("✅ Write succeeded! New schema:")
df_evolved = spark.read.format("delta").load(evolution_path)
df_evolved.printSchema()
print("All rows (notice NULLs for old rows in 'department'):")
df_evolved.show()

### Deep Dive: Schema Evolution Best Practices

**Why schema enforcement is the DEFAULT:**

Imagine a pipeline writes millions of rows daily. One day, the upstream source accidentally renames `revenue` to `rev`. Without enforcement:
* Your table now has both `revenue` (NULLs) and `rev` columns
* All downstream dashboards break
* You don’t notice for days

With enforcement: the pipeline FAILS immediately, you get paged, fix it in minutes.

**When to use mergeSchema:**
* Planned additions: “the vendor is adding a new `phone_number` field next month”
* Bronze/raw tables where you want to accept everything
* Exploratory/staging environments

**When NOT to use mergeSchema:**
* Gold/production tables that dashboards depend on
* Tables with strict SLAs
* Any table where unexpected schema changes indicate a bug

**The medallion approach to schema evolution:**
* **Bronze (raw):** `mergeSchema = true` — accept everything from source
* **Silver (cleaned):** Explicit schema — validate and transform
* **Gold (business):** Strict enforcement — never change without a migration plan

**`overwriteSchema` vs `mergeSchema`:**
* `mergeSchema` = ADD new columns, keep existing
* `overwriteSchema` = REPLACE the entire schema (nuclear option, use with `mode("overwrite")` only)

## 3.6 — Auto Loader (Incremental File Ingestion)

### The Problem

New files land in cloud storage every hour (vendor uploads, API dumps, IoT sensors). You need to:
1. Process only NEW files (don’t re-read old ones)
2. Handle schema changes gracefully
3. Scale to millions of files
4. Be fault-tolerant (exactly-once processing)

### Auto Loader = Databricks’s solution

```python
df_stream = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "json") \
    .option("cloudFiles.schemaLocation", "/path/to/schema") \
    .load("/landing/zone/")

df_stream.writeStream \
    .option("checkpointLocation", "/path/to/checkpoint") \
    .trigger(availableNow=True) \
    .toTable("catalog.schema.target_table")
```

### How it works:
1. **First run:** Discovers all existing files, processes them, remembers what it saw
2. **Next run:** Only processes NEW files that arrived since last run
3. **Checkpoint:** Tracks which files have been processed (fault-tolerant)
4. **Schema inference:** Infers schema from first batch, stores it in `schemaLocation`

### Key Options

| Option | What it does |
| --- | --- |
| `cloudFiles.format` | Source file format (csv, json, parquet, etc.) |
| `cloudFiles.schemaLocation` | Where to store inferred/evolved schema |
| `cloudFiles.inferColumnTypes` | Infer types (default: everything is string) |
| `cloudFiles.schemaEvolutionMode` | `addNewColumns`, `rescue`, `failOnNewColumns`, `none` |

**Note:** Auto Loader uses **Structured Streaming** under the hood. We’ll go deep on streaming in Phase 6. For now, understand the pattern.

In [0]:
# ============================================================
# STEP 9: Auto Loader pattern (simulated with batch for demo)
# ============================================================
# Auto Loader is streaming-based, but we can demo the PATTERN.
# In production you'd use spark.readStream.format("cloudFiles").
# Here we simulate the "process only new files" concept.

from pyspark.sql.functions import col, current_timestamp, input_file_name

landing_path = f"{VOLUME_PATH}/landing_zone"
checkpoint_path = f"{VOLUME_PATH}/checkpoints/autoloader_demo"
target_path = f"{VOLUME_PATH}/processed/autoloader_target"

# --- Simulate: write 3 "batches" of files to landing zone ---
print("Simulating files landing over time...")
print("="*50)

batch1 = spark.createDataFrame([
    (1, "order_A", 100.00, "2024-01-01"),
    (2, "order_B", 200.00, "2024-01-01"),
], ["id", "order_name", "amount", "date"])
batch1.write.mode("overwrite").json(f"{landing_path}/batch_001")
print("Batch 1: 2 orders written to landing zone")

batch2 = spark.createDataFrame([
    (3, "order_C", 150.00, "2024-01-02"),
    (4, "order_D", 300.00, "2024-01-02"),
    (5, "order_E", 250.00, "2024-01-02"),
], ["id", "order_name", "amount", "date"])
batch2.write.mode("overwrite").json(f"{landing_path}/batch_002")
print("Batch 2: 3 orders written to landing zone")

# --- Auto Loader: process all files incrementally ---
print("\nRunning Auto Loader (availableNow=True = process everything then stop):")
print("-"*50)

# This is the REAL Auto Loader pattern:
df_stream = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "json") \
    .option("cloudFiles.inferColumnTypes", "true") \
    .option("cloudFiles.schemaLocation", f"{checkpoint_path}/schema") \
    .load(landing_path)

# Write to a Delta table (trigger once = process all available, then stop)
query = df_stream.writeStream \
    .format("delta") \
    .option("checkpointLocation", f"{checkpoint_path}/data") \
    .trigger(availableNow=True) \
    .start(target_path)

query.awaitTermination()

# Check results
df_result = spark.read.format("delta").load(target_path)
print(f"\n✅ Auto Loader processed {df_result.count()} total rows")
df_result.show()

# --- Now add batch 3 and re-run (only new files processed!) ---
print("\nAdding batch 3 (new files)...")
batch3 = spark.createDataFrame([
    (6, "order_F", 175.00, "2024-01-03"),
    (7, "order_G", 400.00, "2024-01-03"),
], ["id", "order_name", "amount", "date"])
batch3.write.mode("overwrite").json(f"{landing_path}/batch_003")

# Re-run Auto Loader — only processes batch_003!
df_stream2 = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "json") \
    .option("cloudFiles.inferColumnTypes", "true") \
    .option("cloudFiles.schemaLocation", f"{checkpoint_path}/schema") \
    .load(landing_path)

query2 = df_stream2.writeStream \
    .format("delta") \
    .option("checkpointLocation", f"{checkpoint_path}/data") \
    .trigger(availableNow=True) \
    .start(target_path)

query2.awaitTermination()

df_final = spark.read.format("delta").load(target_path)
print(f"✅ After re-run: {df_final.count()} total rows (only new files were processed!)")
df_final.show()

### Deep Dive: Auto Loader Explained

**What `trigger(availableNow=True)` means:**

This is the key for batch-style pipelines. It says: “process ALL files that have arrived since last checkpoint, then STOP.” Perfect for hourly/daily scheduled jobs.

Other trigger options:
* `trigger(once=True)` — same as availableNow but processes as a single batch (less efficient)
* `trigger(processingTime="10 seconds")` — continuous, checks for new files every 10s
* No trigger — continuous streaming (checks as fast as possible)

**Why Auto Loader beats manual `spark.read`:**

| Manual approach | Auto Loader |
| --- | --- |
| Must track which files you’ve already read | Checkpoint tracks automatically |
| Listing millions of files is slow | Uses cloud notifications (instant) |
| Re-running reads everything again | Re-running only reads NEW files |
| Schema changes crash the job | Schema evolution built-in |

**The `_rescued_data` column:**

With schema evolution mode = `rescue` (default), if a new file has unexpected columns, they don’t break your pipeline. Instead they go into a special `_rescued_data` JSON column for you to inspect later.

**Real-world pattern:**
```
/landing/zone/     ← Files land here (CSV from vendor, JSON from API)
      ↓
[Auto Loader]      ← Reads new files only, infers/evolves schema
      ↓
/bronze/table      ← Raw data as Delta (append-only, all history)
      ↓
[Transformation]   ← Clean, validate, enrich
      ↓
/silver/table      ← Cleaned, typed, deduplicated
```

This is the first layer of the **medallion architecture** (covered in Phase 7).

## Phase 3 Checkpoint

**Can you do these?**

- [ ] Explain why Parquet/Delta beats CSV/JSON for analytics (columnar vs row-based)
- [ ] Read a CSV with explicit schema, handle corrupt records with read modes
- [ ] Write data with all 4 save modes and explain when to use each
- [ ] Partition data on disk and explain partition pruning
- [ ] Use `coalesce` vs `repartition` to control output file count
- [ ] Create a Delta table, append data, and use time travel to read previous versions
- [ ] Explain schema enforcement vs schema evolution (`mergeSchema`)
- [ ] Describe the Auto Loader pattern for incremental file ingestion

---

**Key takeaways:**
1. **Always use Delta** in production (Parquet + ACID + time travel)
2. **Explicit schemas** over inferSchema (faster, correct types)
3. **Partition wisely** (low cardinality columns that queries filter on)
4. **Auto Loader** for incremental file processing (not manual `spark.read`)
5. **Schema enforcement** protects production; evolution is opt-in for raw/bronze layers

---

**Next up:** Phase 4 — Delta Lake Deep Dive (MERGE/upserts, OPTIMIZE, Z-ORDER, liquid clustering, VACUUM)